In [0]:
%pip install glow.py
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/84.9 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.9/84.9 kB 4.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


#Create a FASTA index file

In [0]:
from pathlib import Path

fasta_path = "/Volumes/users/nitin_aggarwal/vcf/ruslan/hg38.fa"
fai_path = fasta_path + ".fai"

def build_fai(fasta_file, fai_file):
    with open(fasta_file, "r") as f, open(fai_file, "w") as out:
        offset = 0
        seq_name = None
        seq_len = 0
        line_bases = None
        line_width = None
        
        for line in f:
            line_len = len(line)
            
            if line.startswith(">"):
                if seq_name is not None:
                    out.write(f"{seq_name}\t{seq_len}\t{seq_offset}\t{line_bases}\t{line_width}\n")
                seq_name = line[1:].strip().split()[0]
                seq_len = 0
                seq_offset = offset + line_len
                line_bases = None
                line_width = None
            else:
                bases = len(line.strip())
                if line_bases is None:
                    line_bases = bases
                    line_width = line_len
                seq_len += bases
            
            offset += line_len
        
        if seq_name is not None:
            out.write(f"{seq_name}\t{seq_len}\t{seq_offset}\t{line_bases}\t{line_width}\n")

build_fai(fasta_path, fai_path)

print("FAI index created:", fai_path)

FAI index created: /Volumes/users/nitin_aggarwal/vcf/ruslan/hg38.fa.fai


#Glow split/normalize

In [0]:
# Databricks notebook / job
# VCF (DIRECTORY) -> Glow split/normalize -> Glow PIPE (VCF roundtrip / external tool) -> DELTA tables
# Includes: per-file log table + quarantine table handling.

# ---- Optional install (uncomment if needed)
#%pip install glow.py
#dbutils.library.restartPython()

from datetime import datetime, timezone
import uuid
import traceback

from pyspark.sql import functions as F, types as T

# ----------------------------
# 0) CONFIG
# ----------------------------
VCF_DIR = "/Volumes/users/nitin_aggarwal/vcf/ruslan"
DB = "users.nitin_aggarwal"

TBL_VARIANTS = f"{DB}.variants_flat"
TBL_CALLS    = f"{DB}.variant_calls"
TBL_CSQ      = f"{DB}.variant_consequences_flat"
TBL_FILE_LOG = f"{DB}.vcf_ingest_file_log"

# PIPE-related
PIPE_USE = True
PIPE_CMD = ["/bin/bash", "-lc", "cat"]          # replace "cat" with your annotator command
PIPE_HEADER_MODE = "file"                      # "file" recommended
PIPE_HEADER_VCF  = f"{VCF_DIR}/test_mutect_1.vcf"
PIPE_QUARANTINE_TABLE = f"{DB}.vcf_pipe_quarantine"
PIPE_CLEANUP = True

# Input selection
VCF_SUFFIXES = (".vcf", ".vcf.gz")
MAX_FILES = None
ONLY_THESE_BASENAMES = {"test_mutect_1.vcf", "test_mutect_2.vcf"}  # None to process all

# Glow normalize requires FASTA (+ .fai) accessible to executors
#REF_GENOME_PATH = None
REF_GENOME_PATH = "/Volumes/users/nitin_aggarwal/vcf/ruslan/hg38.fa"

RESET_TABLES_FIRST = True
SKIP_BAD_FILES = True

# Counts/logging
CAPTURE_ROW_COUNTS = True   # does actions(count()); set False for fastest runs

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# ----------------------------
# 0.5) RUN METADATA
# ----------------------------
RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.now(timezone.utc)  # timezone-aware UTC

# ----------------------------
# 1) IMPORT GLOW (fail fast with clear message)
# ----------------------------
try:
    import glow
    glow.register(spark)
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "Glow is not available in this cluster environment.\n"
        "Fix options:\n"
        "  1) Run `%pip install glow.py` then `dbutils.library.restartPython()`\n"
        "  2) Use a Databricks Runtime/cluster image that already includes Glow\n"
        f"Original error: {e}"
    )

# ----------------------------
# 1.5) COPY REFERENCE GENOME TO LOCAL DISK ON ALL NODES
# ----------------------------
import os
import shutil

LOCAL_REF_DIR = "/tmp/ref_genome"
LOCAL_REF_FA  = f"{LOCAL_REF_DIR}/hg38.fa"
LOCAL_REF_FAI = f"{LOCAL_REF_DIR}/hg38.fa.fai"

if REF_GENOME_PATH:
    VOLUME_REF_FA  = REF_GENOME_PATH                          # /Volumes/.../hg38.fa
    VOLUME_REF_FAI = REF_GENOME_PATH + ".fai"                 # /Volumes/.../hg38.fa.fai

    # --- copy on DRIVER ---
    os.makedirs(LOCAL_REF_DIR, exist_ok=True)
    if not os.path.exists(LOCAL_REF_FA):
        print(f"Copying reference FASTA to driver local disk: {LOCAL_REF_FA}")
        shutil.copy(VOLUME_REF_FA, LOCAL_REF_FA)
    if not os.path.exists(LOCAL_REF_FAI):
        print(f"Copying reference index to driver local disk: {LOCAL_REF_FAI}")
        shutil.copy(VOLUME_REF_FAI, LOCAL_REF_FAI)

    # --- copy on EVERY EXECUTOR ---
    def _copy_ref_to_local(iterator):
        """Runs once per partition; copies if not already present."""
        import os, shutil
        os.makedirs(LOCAL_REF_DIR, exist_ok=True)
        if not os.path.exists(LOCAL_REF_FA):
            shutil.copy(VOLUME_REF_FA, LOCAL_REF_FA)
        if not os.path.exists(LOCAL_REF_FAI):
            shutil.copy(VOLUME_REF_FAI, LOCAL_REF_FAI)
        return iterator

    # Touch every executor — use enough partitions to hit all nodes
    _num_parts = max(int(spark.conf.get("spark.executor.instances", "8")) * 4, 16)
    spark.sparkContext.parallelize(range(_num_parts), _num_parts) \
        .mapPartitions(_copy_ref_to_local) \
        .count()

    print(f"Reference genome available on all nodes at: {LOCAL_REF_FA}")

    # OVERRIDE the config so downstream code uses the local path
    REF_GENOME_PATH = LOCAL_REF_FA

# ----------------------------
# 1.6) STAGE PIPE HEADER VCF TO DBFS (Hadoop-accessible) TEMP LOCATION
# ----------------------------
if PIPE_USE and PIPE_HEADER_MODE.lower() == "file":
    _pipe_header_filename = PIPE_HEADER_VCF.rstrip("/").split("/")[-1]
    DBFS_PIPE_HEADER_PATH = f"dbfs:/tmp/pipe_header/{_pipe_header_filename}"

    # Copy via dbutils (Volume -> DBFS), overwrite to keep fresh
    dbutils.fs.mkdirs("dbfs:/tmp/pipe_header")
    dbutils.fs.cp(PIPE_HEADER_VCF, DBFS_PIPE_HEADER_PATH, recurse=False)

    # Override — Glow will resolve this via Hadoop FS (works on all nodes)
    PIPE_HEADER_VCF = DBFS_PIPE_HEADER_PATH
    print(f"Pipe header VCF staged to Hadoop-accessible path: {PIPE_HEADER_VCF}")

# ----------------------------
# 2) EXPLICIT SCHEMAS (avoid CANNOT_DETERMINE_TYPE)
# ----------------------------
CALLS_SCHEMA = T.StructType([
    T.StructField("filename", T.StringType(), True),
    T.StructField("source_file", T.StringType(), True),
    T.StructField("ingest_ts", T.TimestampType(), True),
    T.StructField("variant_id", T.StringType(), True),
    T.StructField("sample_id", T.StringType(), True),
    T.StructField("GT", T.StringType(), True),
    T.StructField("DP", T.IntegerType(), True),
    T.StructField("GQ", T.IntegerType(), True),
    T.StructField("AD", T.StringType(), True),
    T.StructField("PL", T.StringType(), True),
    T.StructField("phased", T.BooleanType(), True),
    T.StructField("AF", T.StringType(), True),
    T.StructField("SB", T.StringType(), True),
    T.StructField("F1R2", T.StringType(), True),
    T.StructField("F2R1", T.StringType(), True),
    T.StructField("PID", T.StringType(), True),
    T.StructField("PGT", T.StringType(), True),
    T.StructField("PS", T.IntegerType(), True),
])

CSQ_SCHEMA = T.StructType([
    T.StructField("filename", T.StringType(), True),
    T.StructField("source_file", T.StringType(), True),
    T.StructField("ingest_ts", T.TimestampType(), True),
    T.StructField("variant_id", T.StringType(), True),
    T.StructField("variant_key", T.StringType(), True),
    T.StructField("chrom", T.StringType(), True),
    T.StructField("pos", T.LongType(), True),
    T.StructField("ref", T.StringType(), True),
    T.StructField("alt", T.StringType(), True),
    T.StructField("csq_index", T.IntegerType(), True),
    T.StructField("CSQ_raw", T.StringType(), True),
])

FILE_LOG_SCHEMA = T.StructType([
    T.StructField("run_id", T.StringType(), False),
    T.StructField("run_ts", T.TimestampType(), False),

    T.StructField("vcf_path", T.StringType(), True),
    T.StructField("filename", T.StringType(), True),

    T.StructField("status", T.StringType(), True),      # processed | skipped
    T.StructField("stage", T.StringType(), True),       # read | split | normalize | pipe | write

    T.StructField("pipe_enabled", T.BooleanType(), True),
    T.StructField("pipe_cmd", T.StringType(), True),
    T.StructField("pipe_header_mode", T.StringType(), True),
    T.StructField("pipe_header_vcf", T.StringType(), True),
    T.StructField("pipe_quarantine_table", T.StringType(), True),

    T.StructField("has_csq", T.BooleanType(), True),
    T.StructField("has_genotypes", T.BooleanType(), True),

    T.StructField("variants_rows", T.LongType(), True),
    T.StructField("calls_rows", T.LongType(), True),
    T.StructField("csq_rows", T.LongType(), True),

    T.StructField("error", T.StringType(), True),
])

# Quarantine schema is not stable across Glow versions; keep it permissive.
# Create an empty table with a single string column so the table exists for reads.
PIPE_QUARANTINE_SCHEMA = T.StructType([
    T.StructField("raw", T.StringType(), True)
])

# ----------------------------
# 3) HELPERS
# ----------------------------
def _dtype_of(df, colname: str):
    for f in df.schema.fields:
        if f.name == colname:
            return f.dataType
    return None

def _is_complex(dt: T.DataType) -> bool:
    return isinstance(dt, (T.ArrayType, T.StructType, T.MapType))

def _safe_col(df, name: str, cast_to: str = None):
    if name in df.columns:
        c = F.col(name)
        return c.cast(cast_to) if cast_to else c
    return F.lit(None).cast(cast_to) if cast_to else F.lit(None)

def _struct_has_field(struct_dt: T.DataType, field_name: str) -> bool:
    return isinstance(struct_dt, T.StructType) and any(f.name == field_name for f in struct_dt.fields)

def _flatten_struct_cols(prefix: str, struct_col: F.Column, struct_dt: T.StructType):
    out = []
    for f in struct_dt.fields:
        nm = f.name
        dt = f.dataType
        full = f"{prefix}{nm}"
        c = struct_col.getField(nm)
        if isinstance(dt, T.StructType):
            out.extend(_flatten_struct_cols(full + "_", c, dt))
        elif isinstance(dt, (T.ArrayType, T.MapType)):
            out.append(F.to_json(c).alias(full))
        else:
            out.append(c.cast("string").alias(full))
    return out

def _list_vcf_files(dir_path: str):
    infos = dbutils.fs.ls(dir_path)
    out = []
    for fi in infos:
        if fi.isDir():
            continue
        p = fi.path
        if any(p.lower().endswith(sfx) for sfx in VCF_SUFFIXES):
            out.append(p)
    return sorted(out)

def _basename(p: str) -> str:
    return p.rstrip("/").split("/")[-1]

def _write_delta(df, table_name: str, mode: str):
    w = df.write.format("delta").mode(mode)
    if mode.lower() == "overwrite":
        w = w.option("overwriteSchema", "true")
    w.saveAsTable(table_name)

def _drop_uc_object_if_exists(fq_name: str):
    parts = fq_name.split(".")
    if len(parts) != 3:
        raise ValueError(f"Expected catalog.schema.name format, got: {fq_name}")

    catalog, schema, name = parts
    rows = spark.sql(f"""
        SELECT table_type
        FROM {catalog}.information_schema.tables
        WHERE table_schema = '{schema}'
          AND table_name = '{name}'
    """).collect()

    if not rows:
        return

    table_type = rows[0]["table_type"]
    if table_type == "VIEW":
        spark.sql(f"DROP VIEW {fq_name}")
        print(f"   - dropped VIEW:  {fq_name}")
    else:
        spark.sql(f"DROP TABLE {fq_name}")
        print(f"   - dropped TABLE: {fq_name}")

def _ensure_table_exists_if_missing(fq_name: str, schema: T.StructType):
    catalog, sch, name = fq_name.split(".")
    exists = spark.sql(f"""
        SELECT 1
        FROM {catalog}.information_schema.tables
        WHERE table_schema='{sch}' AND table_name='{name}'
        LIMIT 1
    """).count() > 0
    if not exists:
        empty_df = spark.createDataFrame([], schema=schema)
        _write_delta(empty_df, fq_name, "overwrite")

def _ensure_empty_table_if_first_write(fq_name: str, schema: T.StructType, mode: str):
    if mode == "overwrite":
        empty_df = spark.createDataFrame([], schema=schema)
        _write_delta(empty_df, fq_name, "overwrite")

def _append_file_log(
    vcf_path: str,
    filename: str,
    status: str,
    stage: str,
    has_csq: bool = None,
    has_genotypes: bool = None,
    variants_rows: int = None,
    calls_rows: int = None,
    csq_rows: int = None,
    error: str = None,
):
    row = [{
        "run_id": RUN_ID,
        "run_ts": RUN_TS,

        "vcf_path": vcf_path,
        "filename": filename,

        "status": status,
        "stage": stage,

        "pipe_enabled": PIPE_USE,
        "pipe_cmd": " ".join(PIPE_CMD) if PIPE_CMD else None,
        "pipe_header_mode": PIPE_HEADER_MODE,
        "pipe_header_vcf": PIPE_HEADER_VCF,
        "pipe_quarantine_table": PIPE_QUARANTINE_TABLE,

        "has_csq": has_csq,
        "has_genotypes": has_genotypes,

        "variants_rows": int(variants_rows) if variants_rows is not None else None,
        "calls_rows": int(calls_rows) if calls_rows is not None else None,
        "csq_rows": int(csq_rows) if csq_rows is not None else None,

        "error": error,
    }]
    spark.createDataFrame(row, schema=FILE_LOG_SCHEMA) \
         .write.format("delta").mode("append").saveAsTable(TBL_FILE_LOG)

def _run_pipe_vcf(df_in):
    if not PIPE_USE:
        return df_in

    if PIPE_HEADER_MODE.lower() == "infer":
        in_vcf_header = "infer"
    elif PIPE_HEADER_MODE.lower() == "file":
        in_vcf_header = PIPE_HEADER_VCF
    else:
        raise ValueError(f"PIPE_HEADER_MODE must be 'infer' or 'file', got: {PIPE_HEADER_MODE}")

    df_out = glow.transform(
        "pipe",
        df_in,
        cmd=PIPE_CMD,
        input_formatter="vcf",
        output_formatter="vcf",
        in_vcf_header=in_vcf_header,
        quarantine_table=PIPE_QUARANTINE_TABLE,
        quarantine_flavor="delta",
        pipe_cleanup=PIPE_CLEANUP
    )
    return df_out

# ----------------------------
# 4) RESET OUTPUT TABLES
# ----------------------------
if RESET_TABLES_FIRST:
    print("RESET_TABLES_FIRST=True -> dropping output objects if they exist...")
    _drop_uc_object_if_exists(TBL_VARIANTS)
    _drop_uc_object_if_exists(TBL_CALLS)
    _drop_uc_object_if_exists(TBL_CSQ)
    _drop_uc_object_if_exists(TBL_FILE_LOG)
    _drop_uc_object_if_exists(PIPE_QUARANTINE_TABLE)

# Ensure log/quarantine tables exist so reads/displays won't fail
_ensure_table_exists_if_missing(TBL_FILE_LOG, FILE_LOG_SCHEMA)
_ensure_table_exists_if_missing(PIPE_QUARANTINE_TABLE, PIPE_QUARANTINE_SCHEMA)

# ----------------------------
# 5) DISCOVER FILES
# ----------------------------
vcf_files = _list_vcf_files(VCF_DIR)
if ONLY_THESE_BASENAMES is not None:
    vcf_files = [p for p in vcf_files if _basename(p) in ONLY_THESE_BASENAMES]
if MAX_FILES is not None:
    vcf_files = vcf_files[:MAX_FILES]

print(f"\nFound {len(vcf_files)} VCF file(s) under {VCF_DIR}")
for p in vcf_files:
    print(" -", p)

if not vcf_files:
    raise ValueError(f"No matching VCF files found under {VCF_DIR}")

print("\nnormalize_variants:", "ENABLED" if REF_GENOME_PATH else "SKIPPED (REF_GENOME_PATH=None)")
print("pipe transformer:", "ENABLED" if PIPE_USE else "DISABLED")
if PIPE_USE:
    print("   - PIPE_CMD:", " ".join(PIPE_CMD))
    print("   - PIPE_HEADER_MODE:", PIPE_HEADER_MODE)
    print("   - PIPE_HEADER_VCF:", PIPE_HEADER_VCF)
    print("   - PIPE_QUARANTINE_TABLE:", PIPE_QUARANTINE_TABLE)
    print("   - PIPE_CLEANUP:", PIPE_CLEANUP)

# ----------------------------
# 6) PROCESS LOOP
# ----------------------------
processed, skipped = 0, 0
mode_variants = "overwrite"
mode_calls    = "overwrite"
mode_csq      = "overwrite"

for vcf_path in vcf_files:
    fn = _basename(vcf_path)
    print(f"\n=== Processing: {vcf_path} ===")

    variants_rows = None
    calls_rows = None
    csq_rows = None

    try:
        # 6.1 Read VCF
        base = spark.read.format("vcf").load(vcf_path)

        # 6.2 Glow transforms (core requirement)
        df = glow.transform("split_multiallelics", base)

        if REF_GENOME_PATH:
            df = glow.transform("normalize_variants", df, reference_genome_path=REF_GENOME_PATH)

        # 6.3 Glow PIPE (VCF serialization/deserialization)
        df = _run_pipe_vcf(df)

        # 6.4 Resolve core columns post-transforms
        chrom_col = "contigName" if "contigName" in df.columns else ("chrom" if "chrom" in df.columns else None)
        start_col = "start" if "start" in df.columns else ("pos" if "pos" in df.columns else None)
        ref_col   = "referenceAllele" if "referenceAllele" in df.columns else ("ref" if "ref" in df.columns else None)
        alts_col  = "alternateAlleles" if "alternateAlleles" in df.columns else ("alt" if "alt" in df.columns else None)
        if chrom_col is None or start_col is None or ref_col is None or alts_col is None:
            raise ValueError(f"Missing core columns. Found: {df.columns}")

        # 6.5 One row per ALT
        variants = (
            df
            .withColumn("chrom", F.col(chrom_col).cast("string"))
            .withColumn("pos", (F.col(start_col) + F.lit(1)).cast("long") if start_col == "start" else F.col(start_col).cast("long"))
            .withColumn("ref", F.col(ref_col).cast("string"))
            .withColumn("alt", F.explode_outer(F.col(alts_col)))
            .withColumn("alt", F.col("alt").cast("string"))
            .withColumn("variant_key", F.concat_ws(":", F.col("chrom"), F.col("pos").cast("string"), F.col("ref"), F.col("alt")))
            .withColumn("variant_id",  F.xxhash64("variant_key").cast("string"))
            .withColumn("source_file", F.lit(vcf_path))
            .withColumn("filename", F.lit(fn))
            .withColumn("ingest_ts", F.current_timestamp())
        )

        has_csq = ("INFO_CSQ" in variants.columns) and isinstance(_dtype_of(variants, "INFO_CSQ"), T.ArrayType)
        has_genotypes = ("genotypes" in variants.columns) and isinstance(_dtype_of(variants, "genotypes"), T.ArrayType)
        print("   - CSQ present?", has_csq)
        print("   - genotypes present?", has_genotypes)

        # ----------------------------
        # 6.6 variants_flat
        # ----------------------------
        info_cols = sorted([c for c in variants.columns if c.startswith("INFO_") and c != "INFO_CSQ"])
        info_exprs = []
        for c in info_cols:
            dt = _dtype_of(variants, c)
            if dt is None:
                continue
            info_exprs.append(F.to_json(F.col(c)).alias(c) if _is_complex(dt) else F.col(c).alias(c))

        core_cols = [
            F.col("filename"),
            F.col("source_file"),
            F.col("ingest_ts"),
            F.col("variant_id"),
            F.col("variant_key"),
            F.col("chrom"),
            F.col("pos"),
            F.col("ref"),
            F.col("alt"),
            (F.to_json(_safe_col(variants, "names"))).alias("names") if "names" in variants.columns else F.lit(None).cast("string").alias("names"),
            F.col("qual").cast("double").alias("qual") if "qual" in variants.columns else F.lit(None).cast("double").alias("qual"),
            (F.to_json(_safe_col(variants, "filters"))).alias("filters") if "filters" in variants.columns else F.lit(None).cast("string").alias("filters"),
            _safe_col(variants, "splitFromMultiAllelic", "boolean").alias("splitFromMultiAllelic"),
        ]

        variants_flat = variants.select(*core_cols, *info_exprs)
        if CAPTURE_ROW_COUNTS:
            variants_flat = variants_flat.persist()
            variants_rows = variants_flat.count()

        _write_delta(variants_flat, TBL_VARIANTS, mode_variants)
        print(f"   - variants_flat: written ({mode_variants})")
        if CAPTURE_ROW_COUNTS:
            variants_flat.unpersist()

        # ----------------------------
        # 6.7 CSQ
        # ----------------------------
        csq_rows = 0
        if has_csq:
            csq_arr_dt = _dtype_of(variants, "INFO_CSQ")
            elem_dt = csq_arr_dt.elementType if isinstance(csq_arr_dt, T.ArrayType) else None

            csq = variants.select(
                "filename", "source_file", "ingest_ts",
                "variant_id", "variant_key", "chrom", "pos", "ref", "alt",
                F.posexplode_outer("INFO_CSQ").alias("csq_index", "csq")
            )

            if isinstance(elem_dt, T.StructType):
                csq_flat_cols = _flatten_struct_cols("CSQ_", F.col("csq"), elem_dt)
                csq_flat = csq.select(
                    "filename", "source_file", "ingest_ts",
                    "variant_id", "variant_key", "chrom", "pos", "ref", "alt", "csq_index",
                    *csq_flat_cols
                )
            else:
                csq_flat = csq.select(
                    "filename", "source_file", "ingest_ts",
                    "variant_id", "variant_key", "chrom", "pos", "ref", "alt", "csq_index",
                    F.col("csq").cast("string").alias("CSQ_raw")
                )

            if CAPTURE_ROW_COUNTS:
                csq_flat = csq_flat.persist()
                csq_rows = csq_flat.count()

            _write_delta(csq_flat, TBL_CSQ, mode_csq)
            print(f"   - variant_consequences_flat: written ({mode_csq})")
            if CAPTURE_ROW_COUNTS:
                csq_flat.unpersist()
        else:
            _ensure_empty_table_if_first_write(TBL_CSQ, CSQ_SCHEMA, mode_csq)
            print("   - variant_consequences_flat: skipped (no INFO_CSQ)")

        # ----------------------------
        # 6.8 Calls
        # ----------------------------
        calls_rows = 0
        if has_genotypes:
            gts_dt = _dtype_of(variants, "genotypes")
            gte_dt = gts_dt.elementType if isinstance(gts_dt, T.ArrayType) else None

            calls = variants.select(
                "filename", "source_file", "ingest_ts",
                "variant_id",
                F.explode_outer("genotypes").alias("g")
            )

            if isinstance(gte_dt, T.StructType):
                sample_id = F.col("g.sampleId").cast("string") if _struct_has_field(gte_dt, "sampleId") else F.lit(None).cast("string")
                phased    = F.col("g.phased").cast("boolean")  if _struct_has_field(gte_dt, "phased") else F.lit(None).cast("boolean")

                DP = (F.col("g.depth") if _struct_has_field(gte_dt, "depth") else (F.col("g.DP") if _struct_has_field(gte_dt, "DP") else F.lit(None))).cast("int").alias("DP")
                GQ = (F.col("g.conditionalQuality") if _struct_has_field(gte_dt, "conditionalQuality") else (F.col("g.GQ") if _struct_has_field(gte_dt, "GQ") else F.lit(None))).cast("int").alias("GQ")

                ADc = F.col("g.alleleDepths") if _struct_has_field(gte_dt, "alleleDepths") else (F.col("g.AD") if _struct_has_field(gte_dt, "AD") else F.lit(None))
                AD  = F.to_json(ADc).alias("AD")

                PLc = (
                    F.col("g.phredLikelihoods") if _struct_has_field(gte_dt, "phredLikelihoods")
                    else (F.col("g.genotypeLikelihoods") if _struct_has_field(gte_dt, "genotypeLikelihoods")
                          else (F.col("g.PL") if _struct_has_field(gte_dt, "PL") else F.lit(None)))
                )
                PL  = F.to_json(PLc).alias("PL")

                if _struct_has_field(gte_dt, "calls"):
                    calls_arr = F.col("g.calls")
                    gt_tokens = F.transform(calls_arr, lambda x: F.when(x < 0, F.lit(".")).otherwise(x.cast("string")))
                    GT = (
                        F.when(calls_arr.isNull(), F.lit(None).cast("string"))
                         .when(phased == True,  F.array_join(gt_tokens, "|"))
                         .otherwise(            F.array_join(gt_tokens, "/"))
                         .alias("GT")
                    )
                else:
                    GT = F.lit(None).cast("string").alias("GT")

                calls_out = calls.select(
                    "filename", "source_file", "ingest_ts",
                    "variant_id",
                    sample_id.alias("sample_id"),
                    GT, DP, GQ, AD, PL,
                    phased.alias("phased")
                )
            else:
                calls_out = spark.createDataFrame([], schema=CALLS_SCHEMA)

            if CAPTURE_ROW_COUNTS:
                calls_out = calls_out.persist()
                calls_rows = calls_out.count()

            _write_delta(calls_out, TBL_CALLS, mode_calls)
            print(f"   - variant_calls: written ({mode_calls})")
            if CAPTURE_ROW_COUNTS:
                calls_out.unpersist()
        else:
            _ensure_empty_table_if_first_write(TBL_CALLS, CALLS_SCHEMA, mode_calls)
            print("   - variant_calls: skipped (no genotypes)")

        # success log
        _append_file_log(
            vcf_path=vcf_path,
            filename=fn,
            status="processed",
            stage="write",
            has_csq=has_csq,
            has_genotypes=has_genotypes,
            variants_rows=variants_rows,
            calls_rows=calls_rows,
            csq_rows=csq_rows,
            error=None
        )

        processed += 1
        mode_variants = "append"
        mode_calls    = "append"
        mode_csq      = "append"

    except Exception as e:
        skipped += 1
        msg = f"{e}\n{traceback.format_exc(limit=20)}"
        print(f"!!! Skipping {vcf_path} due to error:\n{e}")

        _append_file_log(
            vcf_path=vcf_path,
            filename=fn,
            status="skipped",
            stage="unknown",
            error=msg[:32000]
        )

        if not SKIP_BAD_FILES:
            raise

print(f"\nDONE. processed={processed}, skipped={skipped}")
print("Tables:")
print(" -", TBL_VARIANTS)
print(" -", TBL_CALLS)
print(" -", TBL_CSQ)
print(" -", TBL_FILE_LOG)
print(" -", PIPE_QUARANTINE_TABLE)

print("\n=== FILE LOG (this RUN_ID) ===")
display(
    spark.table(TBL_FILE_LOG)
         .where(F.col("run_id") == RUN_ID)
         .select("run_ts","filename","status","stage","pipe_enabled","pipe_cmd",
                 "pipe_header_mode","has_csq","has_genotypes",
                 "variants_rows","calls_rows","csq_rows","error")
         .orderBy(F.col("run_ts").desc(), F.col("filename"))
)

print("\n=== PIPE QUARANTINE (if any) ===")
# Table exists now; it may just be empty
display(spark.table(PIPE_QUARANTINE_TABLE).limit(50))

Copying reference FASTA to driver local disk: /tmp/ref_genome/hg38.fa
Copying reference index to driver local disk: /tmp/ref_genome/hg38.fa.fai
Reference genome available on all nodes at: /tmp/ref_genome/hg38.fa
Pipe header VCF staged to Hadoop-accessible path: dbfs:/tmp/pipe_header/test_mutect_1.vcf
RESET_TABLES_FIRST=True -> dropping output objects if they exist...
   - dropped TABLE: users.nitin_aggarwal.variants_flat
   - dropped TABLE: users.nitin_aggarwal.variant_calls
   - dropped TABLE: users.nitin_aggarwal.variant_consequences_flat
   - dropped TABLE: users.nitin_aggarwal.vcf_ingest_file_log
   - dropped TABLE: users.nitin_aggarwal.vcf_pipe_quarantine

Found 2 VCF file(s) under /Volumes/users/nitin_aggarwal/vcf/ruslan
 - dbfs:/Volumes/users/nitin_aggarwal/vcf/ruslan/test_mutect_1.vcf
 - dbfs:/Volumes/users/nitin_aggarwal/vcf/ruslan/test_mutect_2.vcf

normalize_variants: ENABLED
pipe transformer: ENABLED
   - PIPE_CMD: /bin/bash -lc cat
   - PIPE_HEADER_MODE: file
   - PIPE_HEA

run_ts,filename,status,stage,pipe_enabled,pipe_cmd,pipe_header_mode,has_csq,has_genotypes,variants_rows,calls_rows,csq_rows,error
2026-03-02T15:50:08.481435Z,test_mutect_1.vcf,processed,write,true,/bin/bash -lc cat,file,false,true,3,6,0,null
2026-03-02T15:50:08.481435Z,test_mutect_2.vcf,processed,write,true,/bin/bash -lc cat,file,false,true,3,6,0,null



=== PIPE QUARANTINE (if any) ===


raw


In [0]:
%sql
SELECT COUNT(*) AS variants_rows FROM users.nitin_aggarwal.variants_flat;
SELECT COUNT(*) AS calls_rows    FROM users.nitin_aggarwal.variant_calls;
SELECT COUNT(*) AS csq_rows      FROM users.nitin_aggarwal.variant_consequences_flat;
SELECT COUNT(*) AS log_rows      FROM users.nitin_aggarwal.vcf_ingest_file_log;

log_rows
2


In [0]:
%sql
SELECT filename, COUNT(*) AS n
FROM users.nitin_aggarwal.variants_flat
GROUP BY filename
ORDER BY n DESC;

SELECT filename, COUNT(*) AS n
FROM users.nitin_aggarwal.variant_calls
GROUP BY filename
ORDER BY n DESC;

filename,n
test_mutect_2.vcf,6
test_mutect_1.vcf,6


In [0]:
%sql
SELECT *
FROM users.nitin_aggarwal.vcf_ingest_file_log
ORDER BY run_ts DESC, filename
LIMIT 50;

run_id,run_ts,vcf_path,filename,status,stage,pipe_enabled,pipe_cmd,pipe_header_mode,pipe_header_vcf,pipe_quarantine_table,has_csq,has_genotypes,variants_rows,calls_rows,csq_rows,error
232cda4e-0452-442b-9c18-c0ed2a625936,2026-03-02T15:50:08.481435Z,dbfs:/Volumes/users/nitin_aggarwal/vcf/ruslan/test_mutect_1.vcf,test_mutect_1.vcf,processed,write,true,/bin/bash -lc cat,file,dbfs:/tmp/pipe_header/test_mutect_1.vcf,users.nitin_aggarwal.vcf_pipe_quarantine,false,true,3,6,0,null
232cda4e-0452-442b-9c18-c0ed2a625936,2026-03-02T15:50:08.481435Z,dbfs:/Volumes/users/nitin_aggarwal/vcf/ruslan/test_mutect_2.vcf,test_mutect_2.vcf,processed,write,true,/bin/bash -lc cat,file,dbfs:/tmp/pipe_header/test_mutect_1.vcf,users.nitin_aggarwal.vcf_pipe_quarantine,false,true,3,6,0,null


In [0]:
%sql
SELECT COUNT(*) AS quarantined_rows
FROM users.nitin_aggarwal.vcf_pipe_quarantine;

SELECT *
FROM users.nitin_aggarwal.vcf_pipe_quarantine
LIMIT 50;

raw
